---
format:
  html:
    code-fold: true
jupyter: python3
---

In [15]:
import numpy as np

def power_iteration_buggy(A, num_simulations=100):
    b_k = np.random.rand(A.shape[1])
    for _ in range(num_simulations):
        # Bug 1: Element-wise multiplication instead of matrix multiplication
        b_k1 = A * b_k
        b_k = b_k1 / np.linalg.norm(b_k1)
    return b_k

def pca_buggy(X, num_components):
    # Bug 2: Failing to center the data before computing the covariance matrix
    cov_matrix = np.cov(X, rowvar=False)
    
    eigenvalues, eigenvectors = np.linalg.eigh(cov_matrix)
    sorted_index = np.argsort(eigenvalues)[::-1]
    sorted_eigenvectors = eigenvectors[:, sorted_index]
    eigenvector_subset = sorted_eigenvectors[:, 0:num_components]
    
    X_reduced = np.dot(X, eigenvector_subset)
    return X_reduced

Bug 1 occurs in the power iteration function where element-wise multiplication (*) is used instead of matrix multiplication (@ or np.dot()). Mathematically, power iteration requires projecting the current vector estimate iteratively using the linear transformation defined by matrix $A$, which is expressed as $A \mathbf{v}$. Element-wise multiplication, however, multiplies each column of $A$ by the vector elements, returning a matrix rather than the linearly transformed vector. In NumPy, broadcasting attempts to match the vector to the matrix dimensions, resulting in an entirely incorrect matrix output when calculating the norm. This causes the algorithm to fail silently by returning a matrix instead of the dominant eigenvector, completely breaking the iterative convergence mechanism.

Bug 2 is a conceptual mathematical error in the PCA implementation: it skips the crucial step of mean-centering the data matrix $X$. Mathematically, PCA aims to find the directions of maximum variance. The covariance matrix is defined as the expected value of the outer product of the zero-mean random vectors. If the data is not centered, the computed matrix reflects the second uncentered moment rather than true covariance. Consequently, the first principal component will artificially point toward the mean of the data instead of capturing the axis of highest variance. This severely distorts the dimensionality reduction, leading to a projected subspace that fails to represent the true underlying structure and spread of the dataset.

In [16]:
import numpy as np

def power_iteration_fixed(A, num_simulations=100):
    b_k = np.random.rand(A.shape[1])
    for _ in range(num_simulations):
        # Fix for Bug 1: Use matrix multiplication (@) instead of element-wise (*)
        b_k1 = A @ b_k
        b_k = b_k1 / np.linalg.norm(b_k1)
    return b_k

def pca_fixed(X, num_components):
    # Fix for Bug 2: Mean-center the data before computing the covariance matrix
    X_centered = X - np.mean(X, axis=0)
    cov_matrix = np.cov(X_centered, rowvar=False)
    
    eigenvalues, eigenvectors = np.linalg.eigh(cov_matrix)
    sorted_index = np.argsort(eigenvalues)[::-1]
    sorted_eigenvectors = eigenvectors[:, sorted_index]
    eigenvector_subset = sorted_eigenvectors[:, 0:num_components]
    
    X_reduced = np.dot(X_centered, eigenvector_subset)
    return X_reduced

# --- Test Execution ---
np.random.seed(42)
dummy_matrix = np.random.rand(5, 3)
print("Original Matrix Shape:", dummy_matrix.shape)

# Test PCA
reduced_matrix = pca_fixed(dummy_matrix, num_components=2)
print("PCA Reduced Shape:", reduced_matrix.shape)
print("PCA Output:\n", reduced_matrix)

# Test Power Iteration (requires square matrix)
square_matrix = dummy_matrix.T @ dummy_matrix
dominant_eigenvector = power_iteration_fixed(square_matrix)
print("\nDominant Eigenvector Shape:", dominant_eigenvector.shape)
print("Dominant Eigenvector:\n", dominant_eigenvector)

Original Matrix Shape: (5, 3)
PCA Reduced Shape: (5, 2)
PCA Output:
 [[ 0.5382821  -0.04170504]
 [-0.37801268  0.26959854]
 [ 0.60281427  0.09375913]
 [-0.31232627 -0.5572872 ]
 [-0.45075742  0.23563458]]

Dominant Eigenvector Shape: (3,)
Dominant Eigenvector:
 [0.52458829 0.54271957 0.65594405]


The correction replaces the element-wise multiplication operator (*) with the matrix multiplication operator (@). In linear algebra, computing the next vector in the sequence requires the matrix-vector product $A \mathbf{v}$. NumPy’s @ operator strictly enforces standard matrix multiplication rules between the 2D array and 1D vector. This resolves the broadcasting error, ensuring the output is correctly mapped to a 1D vector at each iteration step, allowing the algorithm to mathematically converge toward the dominant eigenvector.

The correction subtracts the column-wise mean from the dataset $X$ before calculating the covariance matrix. By definition, principal components are the eigenvectors of the data's true covariance matrix, $\Sigma = \frac{1}{n-1} \mathbf{X}^T \mathbf{X}$, which requires $\mathbf{X}$ to be mean-centered. Centering shifts the origin of the coordinate system to the dataset's centroid. This mathematical guarantee ensures the computed eigenvectors correctly identify orthogonal axes of maximum variance, rather than merely pointing toward the uncentered data mean.